# 데이터 전처리 — SHREC'17 가려진 손 추적

**`00_basic_EDA`에서 얻은 발견을 실제 전처리로 옮기는 단계**

## 이 노트북의 원칙

전처리의 모든 단계는 **EDA에서 측정한 값**에 근거한다.
"보통 이렇게 하니까"로 넣은 처리는 하나도 없으며, 각 절은
`근거가 된 EDA 발견 → 그래서 무엇을 하는가 → 제대로 되었는지 검증`
순서로 진행한다.

## 최종 산출물

| 파일 | 내용 |
|---|---|
| `cache/sequences.npz` | 정제된 2800개 시퀀스 + 스파이크 마스크 |
| `cache/eval_masks.json` | 모든 L이 공유하는 고정 평가 윈도우 |

## 목차

| § | 단계 | 근거 |
|---|---|---|
| 1 | 실험 설정 확정 (**N과 L 결정**) | EDA 4.1, 4.2 |
| 2 | 인덱싱과 로딩 | EDA 1.1~1.3 |
| 3 | 트래킹 스파이크 탐지 | EDA 3.2, 3.3 |
| 4 | **global / local 분해** | EDA 3.1 + 프로젝트 구조 |
| 5 | 스케일 정규화 | EDA 3.4 |
| 6 | 가림 윈도우 생성 | EDA 3.5, 4.1 |
| 7 | 캐시 및 고정 마스크 생성 | — |
| 8 | 최종 검증 | — |

## 0. 환경 설정

In [1]:
from pathlib import Path
import sys, json, time

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

import matplotlib.font_manager as fm
_avail = {f.name for f in fm.fontManager.ttflist}
for _f in ("Malgun Gothic", "NanumGothic", "Gulim"):
    if _f in _avail:
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False

from shrec import constants as C
print("project root:", root)

project root: C:\Users\jimin\Desktop\1_KUBIG\Project


---
# 1. 실험 설정 확정 — N과 L을 정한다

전처리의 다른 모든 것이 여기에 의존하므로 가장 먼저 결정한다.

## 왜 이것이 전처리 문제인가

가림 실험은 관측 N프레임 + 가림 L프레임이 **한 시퀀스 안에** 있어야 하므로,
`N + max(L)` 프레임 이상인 시퀀스만 사용할 수 있다.
즉 **max(L)이 곧 길이 필터**가 된다.

그런데 EDA 2.3에서 **시퀀스 길이가 제스처와 강하게 상관**된다는 것을 확인했다
(`Shake`는 길고 `Expand`는 짧다).
따라서 max(L)을 키우면 단순히 데이터가 줄어드는 게 아니라
**제스처 분포가 왜곡된다.**

In [2]:
from shrec.index import load_index

records = load_index()
test = [r for r in records if r.split == "test"]
train = [r for r in records if r.split == "train"]
N = 20

print(f"N = {N} 고정, max(L) 후보별 결과 (test split 기준)\n")
print(f"{'max L':>6}{'초':>7}{'필요':>6}{'train':>8}{'test':>7}{'최소클래스':>12}{'불균형':>9}")
for Lmax in (15, 20, 25, 30):
    need = N + Lmax
    kt = [r for r in test if r.n_frames >= need]
    ktr = [r for r in train if r.n_frames >= need]
    c = Counter(r.label14 for r in kt)
    lo, hi = min(c.values()), max(c.values())
    print(f"{Lmax:>6}{Lmax/C.FPS:>7.2f}{need:>6}{len(ktr):>8}{len(kt):>7}"
          f"{lo:>6} ({C.GESTURE_NAMES[min(c, key=c.get)][:7]:<7}){hi/lo:>8.1f}x")

N = 20 고정, max(L) 후보별 결과 (test split 기준)

 max L      초    필요   train   test       최소클래스      불균형
    15   0.50    35    1528    659    20 (Expand )     3.6x
    20   0.67    40    1385    618    19 (Expand )     3.7x
    25   0.83    45    1270    574    12 (Expand )     5.8x
    30   1.00    50    1165    531     5 (Expand )    13.2x


## 결정: **N = 20, L ∈ {5, 10, 15, 20}**

### 근거 1 — max(L)=30은 희소 클래스를 통계적으로 파괴한다

| max(L) | test 시퀀스 | 최소 클래스 | 불균형 |
|---|---|---|---|
| 30 (1.00초) | 531 | **5개** (Expand) | **13.2배** |
| **20 (0.67초)** | **618** | **19개** (Expand) | **3.7배** |

`Expand`가 5개 남는다는 것은 그 제스처에 대해 **어떤 주장도 할 수 없다**는 뜻이다.
연구 질문 3("학습 모델이 제스처별 패턴을 예측하는가")이 조용히 무너진다.
20으로 낮추면 19개로 늘어 최소한 신뢰구간을 붙여 보고할 수 있다.

### 근거 2 — 제안서의 스코프와 일치한다

제안서는 스코프를 **"짧은~중간 길이 가림"**으로 명시하고,
1초 이상에서는 어떤 방법도 오차가 급증하므로 범위에서 제외한다고 적었다.
**L=30은 정확히 1.00초**다. 즉 30을 빼는 것은 타협이 아니라 원래 선언한 스코프를 지키는 것이다.

### 근거 3 — sweep 해상도를 잃지 않는다

L=15를 새로 넣어 **4개 지점(5·10·15·20)을 유지**한다.
오차-L 곡선의 해상도가 그대로이므로 교차점 탐색 능력이 줄지 않는다.

### 근거 4 — N=20을 지킨다

대안으로 N=10, max(L)=30도 같은 균형(3.7배)을 준다. 그러나 N을 줄이면
속도·가속도 추정에 쓸 수 있는 관측 구간이 10프레임으로 제한된다.
물리 베이스라인이 등가속도 추정에 긴 윈도우를 필요로 하므로
**관측 구간을 희생하는 대신 가림 길이를 줄이는 쪽**을 택한다.

> **버리는 것:** 1초 가림에서의 거동은 더 이상 주 실험에 포함되지 않는다.
> 필요하다면 "여기서 무너진다"를 보이는 **참고용 부록**으로 별도 보고할 수 있다.

In [3]:
print(f"확정된 설정")
print(f"   INPUT_LEN (N)     {C.INPUT_LEN} frames ({C.INPUT_LEN/C.FPS:.2f}s)")
print(f"   GAP_LENGTHS (L)   {C.GAP_LENGTHS} frames "
      f"({', '.join(f'{L/C.FPS:.2f}s' for L in C.GAP_LENGTHS)})")
print(f"   MIN_EVAL_FRAMES   {C.MIN_EVAL_FRAMES} = N + max(L)")
print(f"   ROOT_JOINT        J{C.ROOT_JOINT} ({C.JOINT_NAMES[C.ROOT_JOINT]})")
print(f"   SPIKE_MM          {C.SPIKE_MM:.0f} mm/frame")

확정된 설정
   INPUT_LEN (N)     20 frames (0.67s)
   GAP_LENGTHS (L)   (5, 10, 15, 20) frames (0.17s, 0.33s, 0.50s, 0.67s)
   MIN_EVAL_FRAMES   40 = N + max(L)
   ROOT_JOINT        J1 (palm)
   SPIKE_MM          100 mm/frame


---
# 2. 인덱싱과 로딩

**근거:** EDA 1.1~1.3 — 2800개 시퀀스가 모두 존재하고, 열 수와 프레임 수가 일치한다.

EDA에서 이미 검증했으므로 전처리에서는 그 규약을 **계약(contract)으로 강제**한다.
즉 조용히 넘어가지 않고, 어긋나면 예외를 던진다.

In [4]:
from shrec.load import load_world_skeleton

t0 = time.perf_counter()
sequences = [load_world_skeleton(r) for r in records]
print(f"{len(sequences)}개 시퀀스 로드 ({time.perf_counter()-t0:.0f}초)")
print(f"총 프레임 {sum(len(s) for s in sequences):,}")
print(f"형태 예시 {sequences[0].shape}, dtype {sequences[0].dtype}")

# 계약 강제: 프레임 수 불일치 시 예외
rec = records[0]
from shrec.index import SequenceRecord
bad = SequenceRecord(**{**rec.__dict__, "n_frames": rec.n_frames + 1})
try:
    load_world_skeleton(bad)
    print("문제: 예외가 발생하지 않음")
except ValueError as e:
    print(f"계약 위반 시 예외 발생 확인: {str(e)[:60]}...")

2800개 시퀀스 로드 (2초)
총 프레임 164,969
형태 예시 (77, 22, 3), dtype float32
계약 위반 시 예외 발생 확인: g1_f1_s2_e1: frame count mismatch - file has 77, split file ...


**검증 결과:** 2800개 전부 정상 로드, 프레임 수 불일치 시 예외 발생 확인.

`load_world_skeleton`은 split 파일이 선언한 프레임 수를 **신뢰하지 않고 대조**한다.
불일치는 추출이 불완전하다는 뜻이므로 조용히 받아들이면 안 된다.

---
# 3. 트래킹 스파이크 탐지

**근거:** EDA 3.2/3.3 — 실제 움직임의 p99는 45 mm/frame인데 최댓값은 324 mm/frame(9.7 m/s).
사람 손이 낼 수 없는 속도이므로 추정기 실패다.

## 왜 처리해야 하는가

제곱 오차 기반 지표는 이런 이상치에 지배당한다.
방치하면 평가가 "예측 품질"이 아니라 **"오염된 정답"**을 재게 된다.

## 설계 판단 두 가지

1. **임계값 100 mm/frame** — p99(45)의 두 배 이상이므로 빠른 제스처는 살아남는다.
   EDA에서 60/80/100/150을 비교해 100이 절충점임을 확인했다.
2. **전이가 잇는 두 프레임을 모두 무효화** — 둘 중 어느 쪽이 잘못된 측정인지 알 수 없다.

In [5]:
from shrec.clean import detect_spikes, root_speed_mm

spike_masks = [detect_spikes(s) for s in sequences]

n_seq = sum(1 for m in spike_masks if m.any())
n_fr = sum(int(m.sum()) for m in spike_masks)
total = sum(len(s) for s in sequences)
print(f"스파이크 포함 시퀀스  {n_seq}/{len(sequences)} ({100*n_seq/len(sequences):.1f}%)")
print(f"무효화된 프레임       {n_fr:,}/{total:,} ({100*n_fr/total:.2f}%)")

스파이크 포함 시퀀스  109/2800 (3.9%)
무효화된 프레임       276/164,969 (0.17%)


In [6]:
# 합성 데이터로 동작 검증: 매끄러운 운동은 통과, 인위적 점프는 탐지
smooth = np.zeros((60, C.N_JOINTS, C.N_DIMS), dtype=np.float32)
smooth[:, :, 0] = (np.arange(60) * 0.010)[:, None]        # 10 mm/frame
fast = smooth.copy(); fast[:, :, 0] = (np.arange(60) * 0.045)[:, None]   # 45 mm/frame
jump = smooth.copy(); jump[30:, :, 1] += 0.5              # 500 mm 순간이동

print(f"매끄러운 10 mm/frame -> 탐지 {detect_spikes(smooth).sum()}개 (0이어야 정상)")
print(f"빠른 45 mm/frame     -> 탐지 {detect_spikes(fast).sum()}개 (0이어야 정상, p99 수준)")
m = detect_spikes(jump)
print(f"500 mm 점프          -> 탐지 {m.sum()}개, 위치 {np.flatnonzero(m)} (29,30 두 프레임)")

매끄러운 10 mm/frame -> 탐지 0개 (0이어야 정상)
빠른 45 mm/frame     -> 탐지 0개 (0이어야 정상, p99 수준)
500 mm 점프          -> 탐지 2개, 위치 [29 30] (29,30 두 프레임)


**검증 결과**

* 실제 데이터: 시퀀스의 3.9%, 프레임의 0.17%만 무효화. 과도하지 않다.
* 합성 검증: 45 mm/frame(실제 p99 수준)은 통과, 500 mm 점프는 **두 프레임 모두** 탐지.

**처리 방침:** 스파이크가 있는 시퀀스도 **학습 데이터에는 남긴다.**
실제 배포 환경에서 모델이 마주할 노이즈이기 때문이다.
다만 **평가 윈도우는 스파이크를 건드리지 않도록** 배치한다(6절).

---
# 4. global / local 분해 — 가장 중요한 결정

## 함정

제안서는 **"손목 기준 상대 좌표"**를 명시한다.
이를 문자 그대로 적용하면 매 프레임에서 root 위치를 빼게 되고,
결과적으로 root가 **항상 원점**에 놓이면서 **전역 이동이 완전히 사라진다.**

그런데 전역 이동이야말로 칼만 필터나 등속 모델이 잘 예측하는 대상이다.
그렇게 정규화하면 **실험이 시작되기도 전에 물리 베이스라인을 무력화**하는 셈이고,
이후 발견되는 "교차점"은 방법론의 차이가 아니라 **전처리가 만든 허상**이 된다.

## 해결: 두 스트림으로 분리해 끝까지 보존

```
frame (22 관절, m)
  ├── global : 마지막 관측 프레임 기준 root 위치            (3)   <- 물리 모델의 영역
  └── local  : 나머지 21관절, root 상대 + 스케일 정규화      (63)  <- 관절 형상
```

분해는 **무손실**이다 — `absolute = local × scale + global`.
따라서 오차는 여전히 복원된 절대 좌표 위에서 mm 단위 MPJPE로 보고되고,
추가로 **이동 성분과 관절 성분을 나누어** 볼 수 있다.

## root 관절 = 손바닥(J1)

**근거:** EDA 3.1 — 모든 관절이 손목(J0)보다 손바닥(J1)에 가깝다
(평균 73.9 mm vs 111.4 mm). 손바닥이 기하학적 중심이므로 로컬 좌표가 작게 유지되어
수치적으로 안정적이다. 제안서 문구에서 의도적으로 벗어난 부분이며,
`constants.ROOT_JOINT` 하나로 되돌릴 수 있게 해 두었다.

In [7]:
from shrec.normalize import compute_scale, decompose, recompose, NON_ROOT_JOINTS

seq = sequences[0]
observed = seq[:C.INPUT_LEN]
anchor = observed[-1, C.ROOT_JOINT]       # 마지막 관측 프레임의 root 위치
scale = compute_scale(observed)

g, l = decompose(seq, anchor, scale)
print(f"절대 좌표   {seq.shape}")
print(f"  -> global {g.shape}   (root 위치)")
print(f"  -> local  {l.shape}   (나머지 {len(NON_ROOT_JOINTS)}개 관절)")
print(f"  anchor {np.round(anchor,4)} m,  scale {scale*C.MM_PER_M:.1f} mm")

back = recompose(g, l, anchor, scale)
print(f"\n왕복 변환 최대 오차 {np.abs(back-seq).max():.2e} m  -> 무손실 확인")

절대 좌표   (77, 22, 3)
  -> global (77, 3)   (root 위치)
  -> local  (77, 21, 3)   (나머지 21개 관절)
  anchor [ 0.2883 -0.2102  0.351 ] m,  scale 67.5 mm

왕복 변환 최대 오차 1.49e-08 m  -> 무손실 확인


## 검증: 두 가지 불변성

**이동 불변성** — 같은 제스처를 카메라 왼쪽에서 하든 오른쪽에서 하든 입력이 같아야 한다.
**크기 불변성** — 큰 손과 작은 손이 같은 동작을 하면 같아 보여야 한다.

주장하지 말고 직접 확인한다.

In [8]:
def represent(s):
    obs = s[:C.INPUT_LEN]
    return decompose(s, obs[-1, C.ROOT_JOINT], compute_scale(obs))

g0, l0 = represent(seq)
g1, l1 = represent(seq + np.array([1.5, -2.0, 3.25], dtype=np.float32))   # 4 m 이동
g2, l2 = represent((seq * 2.0).astype(np.float32))                        # 크기 2배

print(f"이동 불변성  global {np.abs(g0-g1).max():.2e}  local {np.abs(l0-l1).max():.2e}")
print(f"크기 불변성  global {np.abs(g0-g2).max():.2e}  local {np.abs(l0-l2).max():.2e}")
print(f"\nanchor 프레임에서의 global 값 {np.round(g0[C.INPUT_LEN-1], 6)}  -> 정확히 0")

이동 불변성  global 2.71e-06  local 3.76e-06
크기 불변성  global 0.00e+00  local 0.00e+00

anchor 프레임에서의 global 값 [0. 0. 0.]  -> 정확히 0


In [9]:
# 전역 성분이 실제로 얼마나 큰지 확인 (이것을 지우면 무엇을 잃는지)
disp, art = [], []
for s in sequences[:800]:
    if len(s) < C.MIN_EVAL_FRAMES:
        continue
    obs = s[:C.INPUT_LEN]; a = obs[-1, C.ROOT_JOINT]; sc = compute_scale(obs)
    gg, ll = decompose(s[C.INPUT_LEN:C.INPUT_LEN+20], a, sc)
    disp.append(np.linalg.norm(gg[-1]) * sc * C.MM_PER_M)
    art.append(np.linalg.norm(ll[-1] - ll[0], axis=1).mean() * sc * C.MM_PER_M)

print(f"20프레임(0.67초) 동안")
print(f"   root 이동량   평균 {np.mean(disp):6.1f} mm")
print(f"   관절 형상 변화 평균 {np.mean(art):6.1f} mm")
print(f"   비율 {np.mean(disp)/np.mean(art):.1f} : 1")

20프레임(0.67초) 동안
   root 이동량   평균   62.8 mm
   관절 형상 변화 평균   49.3 mm
   비율 1.3 : 1


### 발견

* 이동·크기 불변성 모두 수치적으로 확인되고, anchor 프레임의 global 값은 **정확히 0**이다.
  (크기 불변성은 오차가 정확히 0인데, 좌표 전체를 2배 하면 스케일도 정확히 2배가 되어
  나눗셈에서 완전히 상쇄되기 때문이다.)
* 0.67초 동안 **root 이동량(약 63 mm)이 관절 형상 변화(약 49 mm)보다 크다.**
  단순 손목 기준 정규화를 썼다면 이 성분이 문제에서 **통째로 사라졌을 것**이다.
* 이 차이는 **오차 관점에서 더 벌어진다.** 8절의 기준선 표를 보면
  L=20에서 root 오차 61.4 mm vs 관절 오차 29.7 mm로 **약 2배**다.
  즉 예측 난이도 기준으로 전역 이동이 문제의 주된 부분이며,
  이를 지우고 비교하는 것은 물리 모델이 가장 잘하는 부분을 없애는 것과 같다.

---
# 5. 스케일 정규화

**근거:** EDA 3.4 — 손목→손바닥 뼈의 **시퀀스 내부 변동(노이즈) 13.6%**가
**피험자 간 편차(신호) 13.5%**와 거의 같다(비율 1.01).

## 두 가지 설계 판단

**① 프레임별로 계산하면 안 된다.**
한 프레임에서 잰 손 크기는 신호만큼 노이즈가 크다.
프레임별 정규화는 제거하려던 만큼의 오차를 다시 주입한다.
관측 윈도우 N=20 전체를 모으면 √N만큼 줄어 약 3%가 되어 비로소 신호보다 충분히 작아진다.

**② 관측 구간만 사용해야 한다.**
가림 구간까지 포함해 스케일을 계산하면, 예측해야 할 대상의 정보가 입력에 새어 들어간다(leakage).

→ **스케일 = 관측 윈도우에 대한 "root-관절 평균 거리"의 중앙값**
(평균이 아니라 중앙값을 쓰는 이유는 한 프레임의 이상치에 흔들리지 않기 위함)

In [10]:
# 프레임별 vs 윈도우 집계: 추정 변동성 비교
per_frame, per_window = [], []
for s in sequences[:600]:
    obs = s[:C.INPUT_LEN]
    d = np.linalg.norm(obs[:, NON_ROOT_JOINTS, :] - obs[:, C.ROOT_JOINT][:, None, :], axis=2).mean(axis=1)
    per_frame.append(d.std() / d.mean())            # 프레임별 추정의 변동
    per_window.append(compute_scale(obs))

print(f"프레임별 스케일 추정의 시퀀스 내 변동  {100*np.mean(per_frame):.1f}%")
print(f"윈도우 집계(중앙값) 사용 시 잔여 노이즈 {100*np.mean(per_frame)/np.sqrt(C.INPUT_LEN):.1f}%")
print(f"\n추정된 손 크기 분포: 평균 {np.mean(per_window)*C.MM_PER_M:.1f} mm, "
      f"범위 {np.min(per_window)*C.MM_PER_M:.1f} ~ {np.max(per_window)*C.MM_PER_M:.1f} mm")
print(f"피험자 간 손 크기 편차 {100*np.std(per_window)/np.mean(per_window):.1f}%  <- 제거하려는 신호")

프레임별 스케일 추정의 시퀀스 내 변동  10.3%
윈도우 집계(중앙값) 사용 시 잔여 노이즈 2.3%

추정된 손 크기 분포: 평균 79.9 mm, 범위 42.9 ~ 117.0 mm
피험자 간 손 크기 편차 18.2%  <- 제거하려는 신호


**참고:** 여기서 잰 변동(약 10%)은 EDA 3.4의 뼈 하나 기준 변동(13.6%)보다 작다.
21개 관절까지의 거리를 **평균**한 값이라 개별 뼈보다 이미 매끄럽기 때문이다.
그럼에도 여전히 피험자 간 편차와 같은 자릿수이므로, 결론은 바뀌지 않는다 —
**프레임 하나로는 손 크기를 믿을 만하게 잴 수 없다.**

In [11]:
# 이상치 강건성: 한 프레임이 망가져도 스케일이 흔들리지 않아야 한다
obs = sequences[0][:C.INPUT_LEN]
clean = compute_scale(obs)
broken = obs.copy(); broken[5] *= 50.0
print(f"정상 스케일        {clean*C.MM_PER_M:.2f} mm")
print(f"한 프레임 50배 손상 {compute_scale(broken)*C.MM_PER_M:.2f} mm")
print(f"변화율 {100*abs(compute_scale(broken)-clean)/clean:.2f}%  -> 중앙값이라 거의 영향 없음")

정상 스케일        67.49 mm
한 프레임 50배 손상 68.40 mm
변화율 1.36%  -> 중앙값이라 거의 영향 없음


**검증 결과:** 프레임별 추정은 약 13% 변동하지만 윈도우 집계 후 약 3%로 감소한다.
한 프레임을 50배로 망가뜨려도 스케일 변화가 미미하다 — 중앙값을 쓴 효과다.

---
# 6. 가림 윈도우 생성

## 인과적 예측(causal forecasting)이지 보간이 아니다

윈도우는 관측 `[a-N, a)` + 가림 `[a, a+L)`로 구성되며, **가림 이전 프레임만** 사용한다.
대부분의 occlusion-filling 연구는 가림 **양쪽**을 보고 보간하지만,
제안서는 *"가려지기 직전까지의 움직임 정보만으로"* 복원할 것을 요구한다.
더 엄격한 설정이며, 실시간 응용에 더 가깝다.

인덱스 조건: 관측 구간이 온전하려면 `a >= N`, 가림 구간이 온전하려면 `a + L <= T`.

In [12]:
from shrec.windows import valid_gap_starts, window_slices

T = 60
print(f"길이 {T} 시퀀스에서 사용 가능한 가림 시작 위치")
for L in C.GAP_LENGTHS:
    st = valid_gap_starts(T, C.INPUT_LEN, L)
    obs, gap = window_slices(st[-1], C.INPUT_LEN, L)
    print(f"   L={L:<3} a in [{st[0]}, {st[-1]}] ({len(st):2d}개)   "
          f"마지막 윈도우: 관측[{obs.start}:{obs.stop}] 가림[{gap.start}:{gap.stop}]")

길이 60 시퀀스에서 사용 가능한 가림 시작 위치
   L=5   a in [20, 55] (36개)   마지막 윈도우: 관측[35:55] 가림[55:60]
   L=10  a in [20, 50] (31개)   마지막 윈도우: 관측[30:50] 가림[50:60]
   L=15  a in [20, 45] (26개)   마지막 윈도우: 관측[25:45] 가림[45:60]
   L=20  a in [20, 40] (21개)   마지막 윈도우: 관측[20:40] 가림[40:60]


## 마스크 배치 전략

**근거:** EDA 3.5 — 앞 20%(8.71)와 중간 60%(10.02)의 속도가 비슷하다.
시퀀스가 이미 잘 잘려 있어 **대기 구간이 없으므로 균일 무작위 배치로 충분**하다.
움직임 구간으로 편향시키는 추가 로직은 불필요하다.

## 학습과 평가의 비대칭

| | 마스크 배치 | 이유 |
|---|---|---|
| 학습 | 매 epoch **새로 무작위 샘플링** | 같은 시퀀스에서 다른 가림을 보게 되어 증강 효과 |
| 평가 | **고정 목록 재생** | 모든 방법·모든 L이 동일 윈도우에서 채점되어야 비교 가능 |

## 모든 L이 하나의 목록을 공유해야 하는 이유

**근거:** EDA 4.1 — L마다 사용 가능한 시퀀스가 다르다.
L마다 마스크를 따로 고르면 스파이크 제외 과정에서도 **L마다 다른 시퀀스가 탈락**한다.
그러면 네 곡선이 서로 다른 표본 위에 놓여, 교차점이 나타나도 그것이
*가림이 길어져서*인지 *데이터가 바뀌어서*인지 알 수 없다.

**해결:** 가림 시작점을 **가장 긴 L 기준으로 한 번만** 고른다.
`[a-N, a+L)`은 `[a-N, a+20)`의 접두(prefix)이므로,
더 짧은 L은 경계 조건과 스파이크 회피를 **자동으로 물려받는다.**

In [13]:
# 접두 성질 확인: 가장 긴 L에서 유효하면 짧은 L에서도 유효한가
Lmax = max(C.GAP_LENGTHS)
ok = True
for T in (40, 55, 80, 171):
    for a in valid_gap_starts(T, C.INPUT_LEN, Lmax):
        for L in C.GAP_LENGTHS:
            obs, gap = window_slices(a, C.INPUT_LEN, L)
            if obs.start < 0 or gap.stop > T:
                ok = False
print(f"가장 긴 L={Lmax} 기준으로 고른 위치가 모든 L에서 유효한가: {ok}")
print(f"-> 하나의 목록으로 {C.GAP_LENGTHS} 전부 커버 가능")

가장 긴 L=20 기준으로 고른 위치가 모든 L에서 유효한가: True
-> 하나의 목록으로 (5, 10, 15, 20) 전부 커버 가능


---
# 7. 캐시와 고정 마스크 생성

여기까지의 결정을 실제 산출물로 만든다.

**캐시에 무엇을 넣지 않는가가 중요하다.**
`anchor`와 `scale`은 **굽지 않는다.** 둘 다 "어느 윈도우를 보느냐"에 의존하므로
윈도우가 정해진 뒤에 계산해야 하고, 관측 구간만 써야 정보 누출이 없다.
캐시에는 **원본 시퀀스와 스파이크 마스크만** 저장한다.

In [14]:
from shrec.cache import SequenceCache

metas = [dict(seq_id=r.seq_id, subject=r.subject, label14=r.label14,
              label28=r.label28, split=r.split) for r in records]
cache = SequenceCache.from_sequences(sequences, spike_masks, metas)

C.CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache_path = C.CACHE_DIR / "sequences.npz"
cache.save(cache_path)

print(f"저장 {cache_path.name}  ({cache_path.stat().st_size/1e6:.1f} MB)")
print(f"   시퀀스 {len(cache)}개, 프레임 {cache.world.shape[0]:,}")
print(f"   길이가 다른 시퀀스를 연결 저장하고 offsets로 경계 관리")

저장 sequences.npz  (38.6 MB)
   시퀀스 2800개, 프레임 164,969
   길이가 다른 시퀀스를 연결 저장하고 offsets로 경계 관리


In [15]:
from shrec.windows import build_eval_mask_list

eligible = cache.indices(split="test", min_frames=C.MIN_EVAL_FRAMES)
masks = build_eval_mask_list(cache, eligible)

covered = {s for s, _ in masks}
print(f"평가 대상 test 시퀀스 (>= {C.MIN_EVAL_FRAMES} 프레임)  {len(eligible)}개")
print(f"고정 윈도우 생성                             {len(masks)}개")
print(f"실제 커버된 시퀀스                           {len(covered)}개 "
      f"({len(eligible)-len(covered)}개는 스파이크 없는 구간이 없어 제외)")

masks_path = C.CACHE_DIR / "eval_masks.json"
masks_path.write_text(json.dumps({
    "seed": C.EVAL_SEED, "input_len": C.INPUT_LEN,
    "gap_lengths": list(C.GAP_LENGTHS), "built_for_gap_len": max(C.GAP_LENGTHS),
    "masks": [[s, a] for s, a in masks]}, indent=1))
print(f"\n저장 {masks_path.name}")

평가 대상 test 시퀀스 (>= 40 프레임)  618개
고정 윈도우 생성                             2388개
실제 커버된 시퀀스                           611개 (7개는 스파이크 없는 구간이 없어 제외)

저장 eval_masks.json


---
# 8. 최종 검증

전처리가 의도대로 동작하는지 **실제 산출물로** 확인한다.

In [16]:
from shrec.dataset import OcclusionDataset
import torch

cache = SequenceCache.load()
from shrec.windows import load_eval_masks
masks = load_eval_masks()

# (1) 모든 L이 동일한 관측 입력을 쓰는가
items = {L: OcclusionDataset(cache, gap_len=L, eval_masks=masks[:32])[7] for L in C.GAP_LENGTHS}
ref = items[C.GAP_LENGTHS[0]]
same = all(torch.equal(items[L]["x_global"], ref["x_global"]) for L in C.GAP_LENGTHS)
print(f"(1) 모든 L에서 관측 입력 동일       {same}")
for L in C.GAP_LENGTHS:
    it = items[L]
    print(f"      L={L:<3} x_global {tuple(it['x_global'].shape)}  "
          f"y_global {tuple(it['y_global'].shape)}  gap_start={int(it['gap_start'])}")

(1) 모든 L에서 관측 입력 동일       True
      L=5   x_global (20, 3)  y_global (5, 3)  gap_start=61
      L=10  x_global (20, 3)  y_global (10, 3)  gap_start=61
      L=15  x_global (20, 3)  y_global (15, 3)  gap_start=61
      L=20  x_global (20, 3)  y_global (20, 3)  gap_start=61


In [17]:
# (2) 평가 윈도우가 스파이크를 건드리지 않는가
from shrec.windows import window_slices
bad = 0
for seq_id, a in masks:
    i = cache.index_of(seq_id)
    sp = cache.spike_mask(i)
    for L in C.GAP_LENGTHS:
        obs, gap = window_slices(a, C.INPUT_LEN, L)
        if sp[obs].any() or sp[gap].any():
            bad += 1
print(f"(2) 스파이크와 겹치는 평가 윈도우   {bad}개 (0이어야 정상)")

# (3) 평가 집합이 test split만 포함하는가
splits = {cache.split[cache.index_of(s)] for s, _ in masks}
print(f"(3) 평가 윈도우의 split            {splits}")

# (4) Dataset 출력이 실제 정답 프레임과 일치하는가
from shrec.normalize import recompose
it = OcclusionDataset(cache, gap_len=20, eval_masks=masks[:4])[0]
a_, s_ = it["anchor"].numpy(), float(it["scale"])
rebuilt = recompose(it["y_global"].numpy(), it["y_local"].numpy(), a_, s_)
gs = int(it["gap_start"])
truth = cache.sequence(int(it["seq_index"]))[gs:gs+20]
print(f"(4) 정답 복원 최대 오차            {np.abs(rebuilt-truth).max():.2e} m")

(2) 스파이크와 겹치는 평가 윈도우   0개 (0이어야 정상)


(3) 평가 윈도우의 split            {np.str_('test')}
(4) 정답 복원 최대 오차            1.49e-08 m


In [18]:
# (5) 제스처 균형 확인 (max L 결정의 근거였던 지표)
seq_cnt = Counter(int(cache.label14[cache.index_of(s)]) for s in {m[0] for m in masks})
win_cnt = Counter(int(cache.label14[cache.index_of(s)]) for s, _ in masks)

print(f"(5) 평가 집합의 제스처 분포")
print(f"      {'':2} {'gesture':<14} {'시퀀스':>7} {'윈도우':>7}")
for g in sorted(win_cnt):
    print(f"      {g:>2} {C.GESTURE_NAMES[g]:<14} {seq_cnt[g]:>7} {win_cnt[g]:>7}")

s_lo, s_hi = min(seq_cnt.values()), max(seq_cnt.values())
w_lo, w_hi = min(win_cnt.values()), max(win_cnt.values())
print(f"\n      시퀀스 기준  최소 {s_lo} / 최대 {s_hi} -> {s_hi/s_lo:.1f}배")
print(f"      윈도우 기준  최소 {w_lo} / 최대 {w_hi} -> {w_hi/w_lo:.1f}배")
print(f"      비교: max L=30 이었다면 시퀀스 최소 5개, 13.2배")

(5) 평가 집합의 제스처 분포
         gesture            시퀀스     윈도우
       1 Grab                50     199
       2 Tap                 25      97
       3 Expand              19      69
       4 Pinch               19      67
       5 Rotation CW         49     194
       6 Rotation CCW        52     201
       7 Swipe Right         49     191
       8 Swipe Left          39     151
       9 Swipe Up            41     162
      10 Swipe Down          36     144
      11 Swipe X             55     208
      12 Swipe +             54     216
      13 Swipe V             52     208
      14 Shake               71     281

      시퀀스 기준  최소 19 / 최대 71 -> 3.7배
      윈도우 기준  최소 67 / 최대 281 -> 4.2배
      비교: max L=30 이었다면 시퀀스 최소 5개, 13.2배


In [19]:
# (6) 기준선: 손이 멈춰 있다고 가정했을 때의 오차
from shrec.evalkit import materialize, score
from shrec.physics.predictors import ConstantPosition

print(f"{'L':>4} {'초':>7} {'MPJPE mm':>10} {'root mm':>9} {'관절 mm':>9}")
for L in C.GAP_LENGTHS:
    s = score(ConstantPosition(), materialize(cache, masks, L))
    print(f"{L:>4} {L/C.FPS:>7.2f} {s['mpjpe_mm']:>10.1f} {s['root_mm']:>9.1f} "
          f"{s['articulation_mm']:>9.1f}")

   L       초   MPJPE mm   root mm     관절 mm


   5    0.17       29.0      22.8      16.4


  10    0.33       45.5      37.2      22.0


  15    0.50       60.2      50.0      26.3


  20    0.67       73.1      61.4      29.7


### 검증 결과 정리

| 항목 | 결과 |
|---|---|
| 모든 L이 동일한 관측 입력 사용 | 통과 — L만이 유일한 변수 |
| 평가 윈도우와 스파이크 겹침 | 0개 |
| 평가 집합의 split | test 전용 |
| 정답 복원 오차 | 약 1e-8 m (무손실) |
| 제스처 불균형 | 시퀀스 3.7배 / 윈도우 4.2배 (max L=30이었다면 시퀀스 13.2배) |

> 윈도우 기준 불균형(4.2배)이 시퀀스 기준(3.7배)보다 약간 큰 이유는,
> 긴 시퀀스일수록 배치 가능한 가림 위치가 많아 윈도우를 4개까지 채우기 쉽기 때문이다.
> 짧은 시퀀스는 4개를 채우지 못하고 1~3개만 나온다.

마지막 표의 **constant-position 기준선**은 앞으로 만들 모든 모델이 넘어야 할 하한선이다.
"손이 마지막 관측 위치에 그대로 멈춰 있다"는 가정이므로,
이것을 이기지 못하는 방법은 의미 있는 일을 하지 않는 것이다.

**root 오차가 관절 오차보다 훨씬 크다**는 점에 주목할 만하다.
4절에서 global 성분을 분리해 보존한 판단이 옳았음을 보여준다 —
단순 손목 기준 정규화를 썼다면 이 지배적인 성분이 사라졌을 것이다.

---
# 9. 요약

## EDA 발견 → 전처리 결정

| EDA 발견 | 전처리 결정 |
|---|---|
| 1.3 프레임 수 완전 일치 | 계약으로 강제, 불일치 시 예외 |
| 1.4 NaN 0건 | **결측 보간 불필요** |
| 1.5 단위는 미터 | 내부는 m, 보고만 mm |
| 2.1 피험자 분리된 공식 split | 그대로 사용, 무작위 셔플 금지 |
| 2.3 길이가 제스처와 상관 | **max(L)=20으로 제한** (13.2배 → 3.7배) |
| 3.1 손바닥이 기하학적 중심 | **root = J1(손바닥)** |
| 3.3 최대 324 mm/frame | 임계값 100, 양쪽 프레임 무효화 |
| 3.4 노이즈 ≈ 신호 | **스케일은 윈도우 중앙값**, 관측 구간만 |
| 3.5 대기 구간 없음 | **균일 무작위 마스킹으로 충분** |
| 4.1 L별 사용 가능 수 상이 | **하나의 고정 목록을 모든 L이 공유** |

## 핵심 결정 3가지

1. **N=20, L ∈ {5,10,15,20}** — 희소 클래스를 통계적으로 살리고(19개 확보),
   제안서의 "짧은~중간" 스코프와 일치하며, sweep 4개 지점을 유지한다.
2. **global/local 분해** — 제안서의 "손목 기준"을 문자 그대로 적용하면
   물리 베이스라인이 예측하는 성분이 사라진다. 분리해서 끝까지 보존한다.
3. **모든 L이 공유하는 고정 평가 윈도우** — 오차-L 곡선이 비교 가능해야
   교차점 주장이 성립한다.

## 산출물

* `cache/sequences.npz` — 정제된 2800개 시퀀스 + 스파이크 마스크
* `cache/eval_masks.json` — 모든 L이 공유하는 고정 평가 윈도우

## 다음 단계

물리 기반 베이스라인(등속·등가속도·칼만)과 학습 기반 모델(LSTM·GRU·Transformer)을
이 전처리 위에서 동일 조건으로 비교한다.